In [1]:
import os
import re
import numpy as np
import pandas as pd
import torch
import nltk

from tqdm import tqdm
from datasets import Dataset
from rouge_score import rouge_scorer, scoring

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)

from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer

nltk.download('punkt')

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-03-20 19:40:03.161574: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 19:40:04.062254: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions

True

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
DATA_PATH = "../Data/news.tsv"

df = pd.read_csv(DATA_PATH, sep="\t")

df = df.sample(n=10000, random_state=42).reset_index(drop=True)

AUTO DETECT TEXT COLUMN

In [4]:
FIELDS = ["News body", "body", "text", "content", "article", "news"]

text_col = None
for c in FIELDS:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    str_cols = [c for c in df.columns if df[c].dtype == "object"]
    avg_lengths = {c: df[c].astype(str).str.len().mean() for c in str_cols}
    text_col = max(avg_lengths, key=avg_lengths.get)

print(f"Using column: {text_col}")

df["text"] = df[text_col].astype(str)

Using column: News body


Text Cleaning

In [5]:
def clean_text(t):
    t = str(t)
    t = re.sub(r"\S+@\S+", " ", t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

df["text"] = df["text"].fillna("").apply(clean_text)

df = df[df["text"].str.len() > 80].reset_index(drop=True)

GENERATE SUMMARIES (TextRank)

In [6]:
def auto_summary(text, num_sentences=4):
    parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
    summarizer = TextRankSummarizer()
    sentences = summarizer(parser.document, num_sentences)
    return " ".join(str(s) for s in sentences)

print("Generating summaries...")
df["summary"] = df["text"].apply(auto_summary)

df = df[df["summary"].str.len() > 40].reset_index(drop=True)

Generating summaries...


Training Dataset

In [7]:
raw_dataset = Dataset.from_pandas(df[["text", "summary"]])
raw_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

Models

In [8]:
MODEL_CONFIGS = [
    {"name": "facebook/bart-large-cnn", "label": "BART-Large-CNN"},
    {"name": "t5-small", "label": "T5-Small"}
]

Preprocess Function

In [9]:
def preprocess(batch, tokenizer, model_name):

    inputs = batch["text"]
    targets = batch["summary"]

    if "t5" in model_name:
        inputs = ["summarize: " + x for x in inputs]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        padding="max_length",
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=160,
        padding="max_length",
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

TRAIN FUNCTION

In [14]:
def train_model(model_name, label):

    print(f"\nTraining {label}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    tokenized = raw_dataset.map(
        lambda x: preprocess(x, tokenizer, model_name),
        batched=True,
        remove_columns=raw_dataset["train"].column_names
    )

    training_args = TrainingArguments(
        output_dir=f"./{label}_checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        fp16=True,
        evaluation_strategy="epoch",
        logging_steps=200,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"]
    )

    trainer.train()

    save_path = f"./{label}_model"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    return save_path

Summary Generation Function

In [15]:
def generate_summary(model, tokenizer, text, model_name):

    if "t5" in model_name:
        text = "summarize: " + text

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=180,
            min_length=40,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=2.0
        )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

Evaluation

In [16]:
def evaluate_model(model_path, model_name, label):

    print(f"\nEvaluating {label}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    preds, refs = [], []

    MAX_EVAL = min(300, len(raw_dataset["test"]))

    for i in tqdm(range(MAX_EVAL)):
        sample = raw_dataset["test"][i]

        pred = generate_summary(
            model,
            tokenizer,
            sample["text"],
            model_name
        )

        preds.append(pred)
        refs.append(sample["summary"])

    scorer = rouge_scorer.RougeScorer(
        ['rouge1','rouge2','rougeL','rougeLsum'],
        use_stemmer=True
    )

    agg = scoring.BootstrapAggregator()

    for r, p in zip(refs, preds):
        agg.add_scores(scorer.score(r, p))

    res = agg.aggregate()

    return {
        "Model": label,
        "rouge1": res['rouge1'].mid.fmeasure,
        "rouge2": res['rouge2'].mid.fmeasure,
        "rougeL": res['rougeL'].mid.fmeasure,
        "rougeLsum": res['rougeLsum'].mid.fmeasure,
    }

In [17]:
all_results = []

for config in MODEL_CONFIGS:

    model_path = train_model(config["name"], config["label"])

    result = evaluate_model(
        model_path,
        config["name"],
        config["label"]
    )

    result["Average Score"] = np.mean([
        result["rouge1"],
        result["rouge2"],
        result["rougeL"],
        result["rougeLsum"]
    ])

    all_results.append(result)


Training BART-Large-CNN


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/8658 [00:00<?, ? examples/s]

Map:   0%|          | 0/963 [00:00<?, ? examples/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


...

Map:   0%|          | 0/963 [00:00<?, ? examples/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
0,0.990100,0.796561
1,0.890800,0.776082



Evaluating T5-Small


100%|██████████| 300/300 [09:47<00:00,  1.96s/it]


In [18]:
OUTPUT_CSV = "../Summarization/summarization_comparison_results.csv"

df_results = pd.DataFrame(all_results).round(4)

if os.path.exists(OUTPUT_CSV):
    old_df = pd.read_csv(OUTPUT_CSV)
    df_results = pd.concat([old_df, df_results], ignore_index=True)

df_results = df_results.drop_duplicates(subset=["Model"], keep="last")
df_results = df_results.sort_values(by="Average Score", ascending=False)

df_results.to_csv(OUTPUT_CSV, index=False)

In [19]:
print("\nComparison of all Models based on F1 score")
print(df_results)


Comparison of all Models based on F1 score
            Model  rouge1  rouge2  rougeL  rougeLsum  Average Score
3  BART-Large-CNN  0.6921  0.6096  0.6255     0.6252         0.6381
4        T5-Small  0.5218  0.4052  0.4442     0.4447         0.4539
0        TextRank  0.5028  0.3924  0.4402     0.4402         0.4451
1          TF-IDF  0.4836  0.3779  0.4144     0.4144         0.4253
2    Seq2Seq LSTM  0.1448  0.0406  0.1343     0.1339         0.1134


In [20]:
best = df_results.iloc[0]

print("\nBEST MODEL")
print("Model Name      :", best["Model"])
print("ROUGE-1         :", best["rouge1"])
print("ROUGE-2         :", best["rouge2"])
print("ROUGE-L         :", best["rougeL"])
print("ROUGE-Lsum      :", best["rougeLsum"])
print("Average Score   :", best["Average Score"])


BEST MODEL
Model Name      : BART-Large-CNN
ROUGE-1         : 0.6921
ROUGE-2         : 0.6096
ROUGE-L         : 0.6255
ROUGE-Lsum      : 0.6252
Average Score   : 0.6381
